# LightOnOCR-2 Magyar Fine-tuning (v11)

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Telepítés + Fontok
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow opencv-python-headless

!mkdir -p /content/fonts

print('Liberation fontok...')
!wget -q 'https://github.com/liberationfonts/liberation-fonts/files/7261482/liberation-fonts-ttf-2.1.5.tar.gz' -O /tmp/liberation.tar.gz
!tar -xzf /tmp/liberation.tar.gz -C /tmp/
!cp /tmp/liberation-fonts-ttf-2.1.5/*.ttf /content/fonts/

print('DejaVu fontok...')
!wget -q 'https://github.com/dejavu-fonts/dejavu-fonts/releases/download/version_2_37/dejavu-fonts-ttf-2.37.zip' -O /tmp/dejavu.zip
!unzip -q -o /tmp/dejavu.zip -d /tmp/
!cp /tmp/dejavu-fonts-ttf-2.37/ttf/*.ttf /content/fonts/

print('FreeFonts...')
!wget -q 'https://ftp.gnu.org/gnu/freefont/freefont-ttf-20120503.zip' -O /tmp/freefont.zip
!unzip -q -o /tmp/freefont.zip -d /tmp/
!cp /tmp/freefont-20120503/*.ttf /content/fonts/

!ls /content/fonts/

In [ ]:
# 2. Font teszt
import os, glob
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from IPython.display import display

def test_font(fp):
    try:
        font = ImageFont.truetype(fp, 32)
        img = Image.new('RGB', (400, 50), 'white')
        ImageDraw.Draw(img).text((10, 8), 'őűŐŰ öüóőúéáűí', fill='black', font=font)
        return np.sum(np.array(img) < 100) > 100, img
    except: return False, None

FONTS = []
for fp in sorted(glob.glob('/content/fonts/*.ttf')):
    name = os.path.basename(fp).replace('.ttf', '')
    ok, img = test_font(fp)
    if ok:
        FONTS.append((name, fp))
        print(f'✓ {name}')
        if len(FONTS) <= 4: display(img)

print(f'\n{len(FONTS)} font')

In [ ]:
# 3. Augmentációk
import cv2, random
from PIL import ImageFilter
from pathlib import Path

def add_noise(img):
    arr = np.array(img).astype(np.float32)
    return Image.fromarray(np.clip(arr + np.random.normal(0, 5, arr.shape), 0, 255).astype(np.uint8))

def rotate_img(img):
    return img.rotate(random.uniform(-2, 2), fillcolor='white')

def apply_aug(img):
    if random.random() < 0.3: img = add_noise(img)
    if random.random() < 0.4: img = rotate_img(img)
    if random.random() < 0.2: img = img.filter(ImageFilter.GaussianBlur(0.5))
    return img

print('✓ Augmentációk')

In [ ]:
# 4. Adatgenerálás
import json

IMG_W, IMG_H = 800, 400

WORDS = ['őr','őriz','ők','ősz','ősi','őszinte','erő','idő','mező','tető','fő','nő','bő',
         'belső','külső','felső','alsó','utolsó','első','költő','festő','vezető','börtön',
         'Győr','dőlt','töröl','pörög','görög','örök','űr','űrlap','gyűrű','tűz','fűz',
         'gyűjt','tűnik','fűszer','hűtő','hűvös','hűség','szürke','szűk','szűr','sűrű',
         'bűvös','működik','műszer','Csatornadíj','vízdíj','tükörfúrógép','árvíztűrő',
         'fizetendő','összeg','adószám','határidő']

def gen_text():
    lines = [' '.join(random.sample(WORDS, 6))]
    lines.append(f'Fizetendő összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft')
    lines.append(f'Csatornadíj: {random.randint(1,9)} {random.randint(100,999):03d} Ft')
    lines.append(f'Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}')
    lines.append('öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ')
    lines.append('Árvíztűrő tükörfúrógép')
    lines.append(' '.join(random.sample(WORDS, 5)))
    return '\n'.join(lines)

def render(text, font_path, size=24):
    img = Image.new('RGB', (IMG_W, IMG_H), 'white')
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_path, size)
    y = 30
    for line in text.split('\n'):
        draw.text((30, y), line, fill='black', font=font)
        y += int(size * 1.4)
    return img

Path('data/images').mkdir(parents=True, exist_ok=True)
annotations = []
N = 600

print(f'Generálás: {N} kép...')
for i in range(N):
    text = gen_text()
    fname, fpath = random.choice(FONTS)
    img = render(text, fpath, random.choice([20,22,24,26]))
    if random.random() < 0.5: img = apply_aug(img)
    img.save(f'data/images/{i:05d}.png')
    annotations.append({'image': f'{i:05d}.png', 'text': text})
    if (i+1) % 100 == 0: print(f'  {i+1}/{N}')

with open('data/annotations.jsonl', 'w', encoding='utf-8') as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + '\n')

print(f'✓ {N} kép')
display(Image.open('data/images/00000.png'))

In [ ]:
# 5. Modell
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

MODEL_ID = 'lightonai/LightOnOCR-2-1B-base'
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','k_proj','v_proj','o_proj'], lora_dropout=0.05))
model.print_trainable_parameters()

In [ ]:
# 6. Dataset
from torch.utils.data import Dataset as TorchDataset

class OCRDataset(TorchDataset):
    def __init__(self, jsonl_path, img_dir, processor):
        self.processor = processor
        self.img_dir = img_dir
        self.data = []
        with open(jsonl_path, encoding='utf-8') as f:
            for line in f:
                self.data.append(json.loads(line))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(f"{self.img_dir}/{item['image']}").convert('RGB')
        
        img_in = self.processor.image_processor(img, return_tensors='pt')
        txt_in = self.processor.tokenizer(item['text'], return_tensors='pt', padding='max_length', max_length=512, truncation=True)
        
        return {
            'pixel_values': img_in['pixel_values'].squeeze(0),
            'input_ids': txt_in['input_ids'].squeeze(0),
            'attention_mask': txt_in['attention_mask'].squeeze(0),
            'labels': txt_in['input_ids'].squeeze(0),
        }

dataset = OCRDataset('data/annotations.jsonl', 'data/images', processor)
print(f'✓ {len(dataset)} kép')
print(f'Pixel shape: {dataset[0]["pixel_values"].shape}')

In [ ]:
# 7. Training
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./lora',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    logging_steps=20,
    save_steps=100,
    bf16=True,
    remove_unused_columns=False,
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
print('Tanítás...')
trainer.train()
print('✓ Kész!')

In [ ]:
# 8. Mentés
model.save_pretrained('./lora')
merged = model.merge_and_unload()
merged.save_pretrained('./merged')
processor.save_pretrained('./merged')
print('✓ Mentve')

In [ ]:
# 9. Teszt
for idx in [0, 100, 200]:
    img = Image.open(f'data/images/{idx:05d}.png')
    inputs = processor.image_processor(img, return_tensors='pt')
    inputs = {k: v.to(merged.device) for k, v in inputs.items()}
    inputs['input_ids'] = processor.tokenizer('', return_tensors='pt')['input_ids'].to(merged.device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=400, do_sample=False)
    print(f'\n=== #{idx} ===')
    display(img)
    print(processor.tokenizer.decode(out[0], skip_special_tokens=True)[:300])

In [ ]:
# 10. Letöltés
!zip -r merged.zip merged/
from google.colab import files
files.download('merged.zip')
print('\nMAC: unzip merged.zip && mlx_vlm convert --hf-path merged --mlx-path lighton-hun-mlx -q --q-bits 4')